# 03 — Node2Vec Embeddings

Generates and analyses node2vec embeddings for bird social networks.  
Uses the `node2vec` package (gensim-backed) — **no StellarGraph / TensorFlow needed**,
fully compatible with Apple Silicon.

Steps:
1. Train embeddings on an edge list  
2. Visualise with UMAP (or PCA fallback)  
3. Cluster embeddings (KMeans + HDBSCAN)  
4. Compare clusters to known labels (sex)  
5. Compare day vs. night embeddings  

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import OUTPUT_ROOT
from embeddings.node2vec_pipeline import run_context, load_graph
from ml.features import build_feature_matrix, load_sex_labels
from ml.clustering import kmeans_sweep, hdbscan_cluster, reduce_for_plot, evaluate_clustering

YEAR   = 2016
PLOT   = 'SPRA'
PERIOD = 'daytime'

## 1. Generate embeddings

In [ ]:
# If embeddings already exist in workspace, this loads them from disk.
# To re-generate, delete the embeddings_*.csv file first.
label   = f'{YEAR}_{PLOT}_{PERIOD}'
emb_path = OUTPUT_ROOT / label / f'embeddings_{label}.csv'

if emb_path.exists():
    print('Loading cached embeddings...')
    emb_df = pd.read_csv(emb_path)
else:
    print('Training node2vec (this may take ~1 min)...')
    emb_df = run_context(YEAR, PLOT, PERIOD)

print(f'Embeddings shape: {emb_df.shape}  ({emb_df.shape[0]} birds × {emb_df.shape[1]-1} dims)')
emb_df.head()

## 2. Build feature matrix & reduce for visualisation

In [ ]:
# Embeddings only (pure node2vec)
X_emb, feat_names_emb, meta = build_feature_matrix(
    YEAR, PLOT, PERIOD,
    use_graph_metrics=False,
    use_embeddings=True,
)
print(f'X shape (embeddings only): {X_emb.shape}')

# Combined: graph metrics + embeddings
X_all, feat_names_all, meta = build_feature_matrix(
    YEAR, PLOT, PERIOD,
    use_graph_metrics=True,
    use_embeddings=True,
)
print(f'X shape (combined):        {X_all.shape}')
meta.head()

In [ ]:
# 2D projection for scatter plots
X2d = reduce_for_plot(X_emb, method='umap')   # falls back to PCA if umap not installed
print('2D projection shape:', X2d.shape)

## 3. KMeans clustering

In [ ]:
labels_km, summary_km = kmeans_sweep(X_emb)
print('KMeans best k:', summary_km['best_k'])
print('Silhouette:   ', summary_km['silhouette'])

# Elbow + silhouette plot
ks = sorted(summary_km['k_scores'])
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(ks, [summary_km['k_inertias'][k] for k in ks], 'o-')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow')
axes[1].plot(ks, [summary_km['k_scores'][k] for k in ks], 'o-', color='coral')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette'); axes[1].set_title('Silhouette')
plt.tight_layout(); plt.show()

In [ ]:
# Scatter plot coloured by KMeans cluster
fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(X2d[:, 0], X2d[:, 1], c=labels_km, cmap='tab10', s=30, alpha=0.8)
plt.colorbar(scatter, ax=ax, label='KMeans cluster')
ax.set_title(f'Embedding space — {YEAR} {PLOT} {PERIOD}\nKMeans k={summary_km["best_k"]}')
ax.set_xlabel('UMAP dim 1'); ax.set_ylabel('UMAP dim 2')
plt.tight_layout(); plt.show()

## 4. HDBSCAN clustering

In [ ]:
labels_hdb, summary_hdb = hdbscan_cluster(X_emb)
print('HDBSCAN clusters:', summary_hdb['n_clusters'])
print('Noise points:    ', summary_hdb['n_noise'])
print('Silhouette:      ', summary_hdb['silhouette'])

fig, ax = plt.subplots(figsize=(7, 6))
colors = ['grey' if l == -1 else l for l in labels_hdb]
ax.scatter(X2d[:, 0], X2d[:, 1], c=labels_hdb, cmap='tab10', s=30, alpha=0.8)
ax.set_title(f'HDBSCAN — {summary_hdb["n_clusters"]} clusters, {summary_hdb["n_noise"]} noise')
ax.set_xlabel('UMAP dim 1'); ax.set_ylabel('UMAP dim 2')
plt.tight_layout(); plt.show()

## 5. Cluster vs. known labels

In [ ]:
if 'sex' in meta.columns and meta['sex'].notna().sum() > 0:
    score_km = evaluate_clustering(labels_km, meta['sex'])
    print('KMeans vs sex label:  ', score_km)
    score_hdb = evaluate_clustering(labels_hdb, meta['sex'])
    print('HDBSCAN vs sex label: ', score_hdb)

    # Colour scatter by sex
    sex_enc, _ = pd.factorize(meta['sex'])
    fig, ax = plt.subplots(figsize=(7, 6))
    for label, color in [(0, 'steelblue'), (1, 'coral'), (-1, 'grey')]:
        mask = sex_enc == label
        name = meta['sex'].iloc[mask.nonzero()[0][0]] if mask.any() else str(label)
        ax.scatter(X2d[mask, 0], X2d[mask, 1], label=name, s=30, alpha=0.8)
    ax.legend(title='Sex')
    ax.set_title(f'Embedding coloured by sex — {YEAR} {PLOT} {PERIOD}')
    ax.set_xlabel('UMAP dim 1'); ax.set_ylabel('UMAP dim 2')
    plt.tight_layout(); plt.show()
else:
    print('Sex labels not available for this context.')

## 6. Day vs. Night embedding comparison

In [ ]:
X_day, _, meta_day = build_feature_matrix(YEAR, PLOT, 'daytime',   use_graph_metrics=False, use_embeddings=True)
X_ngt, _, meta_ngt = build_feature_matrix(YEAR, PLOT, 'nighttime', use_graph_metrics=False, use_embeddings=True)

# Project jointly
X_joint = np.vstack([X_day, X_ngt])
period_labels = ['day'] * len(X_day) + ['night'] * len(X_ngt)
X2d_joint = reduce_for_plot(X_joint)

fig, ax = plt.subplots(figsize=(8, 6))
colors = {'day': 'steelblue', 'night': 'coral'}
for period in ['day', 'night']:
    mask = [p == period for p in period_labels]
    ax.scatter(X2d_joint[mask, 0], X2d_joint[mask, 1],
               label=period, color=colors[period], alpha=0.6, s=25)
ax.legend(title='Period')
ax.set_title(f'Day vs. Night embedding space — {YEAR} {PLOT}')
ax.set_xlabel('UMAP dim 1'); ax.set_ylabel('UMAP dim 2')
plt.tight_layout(); plt.show()